In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ModuleNotFoundError: No module named 'kagglehub'

In [2]:
import pandas as pd
import numpy as np

In [10]:
movie=pd.read_csv("TMDB_all_movies.csv")

In [11]:
movie.head(5)

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,budget,imdb_id,...,spoken_languages,cast,director,director_of_photography,writers,producers,music_composer,imdb_rating,imdb_votes,poster_path
0,2,Ariel,7.106,371.0,Released,1988-10-21,0.0,73.0,0.0,tt0094675,...,suomi,"Kari Helaseppä, Jaakko Talaskivi, Mikko Remes,...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Aki Kaurismäki,NaN,7.4,9791.0,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg
1,3,Shadows in Paradise,7.300,439.0,Released,1986-10-17,0.0,74.0,0.0,tt0092149,...,"svenska, suomi, English","Jukka-Pekka Palo, Svante Korkiakoski, Mari Ran...",Aki Kaurismäki,Timo Salminen,Aki Kaurismäki,Mika Kaurismäki,NaN,7.4,8676.0,/nj01hspawPof0mJmlgfjuLyJuRN.jpg
2,5,Four Rooms,5.905,2851.0,Released,1995-12-09,4257354.0,98.0,4000000.0,tt0113101,...,English,"Sammi Davis, Salma Hayek Pinault, Lawrence Ben...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Phil Parmet, Guillermo Navarro, Rodrigo García...","Quentin Tarantino, Robert Rodriguez, Alexandre...","Quentin Tarantino, Lawrence Bender, Alexandre ...",Combustible Edison,6.7,116968.0,/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg
3,6,Judgment Night,6.500,370.0,Released,1993-10-15,12136938.0,109.0,21000000.0,tt0107286,...,English,"Jeremy Piven, Lydell M. Cheshier, Michael DeLo...",Stephen Hopkins,Peter Levy,"Lewis Colick, Jere Cunningham","Lloyd Segan, Gene Levy, Marilyn Vance",Alan Silvestri,6.6,21111.0,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg
4,8,Life in Loops (A Megacities RMX),7.200,30.0,Released,2006-01-01,0.0,80.0,42000.0,tt0825671,...,"English, हिन्दी, 日本語, Pусский, Español",NaN,Timo Novotny,Wolfgang Thaler,"Timo Novotny, Michael Glawogger","Timo Novotny, Ulrich Gehmacher",NaN,8.1,285.0,/7ln81BRnPR2wqxuITZxEciCe1lc.jpg


In [12]:
movie.columns

Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'budget', 'imdb_id', 'original_language',
       'original_title', 'overview', 'popularity', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'cast', 'director', 'director_of_photography', 'writers', 'producers',
       'music_composer', 'imdb_rating', 'imdb_votes', 'poster_path'],
      dtype='object')

In [13]:
# Split the strings into lists of countries
movie["production_countries_split"] = movie["production_countries"].str.split(",")

# Explode the lists into separate rows
movie_exploded = movie.explode("production_countries_split")

# Strip whitespace (if any)
movie_exploded["production_countries_split"] = (
    movie_exploded["production_countries_split"].str.replace(" ", "")
)

# Now get top 15 most frequent countries
langs = (
    movie_exploded["production_countries_split"]
    .value_counts()
    .head(15)
    .index.tolist()
)
movie = movie.drop(columns=["production_countries_split"])
print(langs)


['UnitedStatesofAmerica', 'France', 'UnitedKingdom', 'Japan', 'Germany', 'Canada', 'India', 'Italy', 'Brazil', 'Spain', 'Mexico', 'Russia', 'China', 'SovietUnion', 'Argentina']


In [14]:
type(movie["production_countries"])
# Output: <class 'pandas.core.series.Series'>

pandas.core.series.Series

In [15]:
movie = movie.drop(columns=['status','runtime','tagline','spoken_languages','director_of_photography','music_composer','writers', 'producers','original_title'])

In [16]:
type(movie['release_date'])

pandas.core.series.Series

In [17]:
# Safe function for decade extraction
def fetch_year_decade(date):
    if pd.isna(date):          # handle NaN
        return None
    date = str(date)           # ensure string
    parts = date.split("-")
    return parts[0][:3]       # decade prefix

def cast_fetching(cast_list):
    # example: keep only first 3 cast members
    if pd.isna(cast_list):
        return None
    return cast_list[:3]

def imdb_score(rating, votes):
    # simple weighted score example
    return rating * np.log1p(votes)

def financial_group(revenue, budget):
    if pd.isna(revenue) or pd.isna(budget):
        return None
    if budget == 0 :
        return "None"
    profit_pct = (revenue - budget) / budget * 100
    # map to your tag ranges here
    if profit_pct < -50: return "Disaster"
    elif profit_pct < 0: return "Flop"
    elif profit_pct <= 10: return "Average"
    elif profit_pct <= 25: return "Semi-Hit"
    elif profit_pct <= 50: return "Hit"
    elif profit_pct <= 100: return "Super Hit"
    elif profit_pct <= 200: return "Blockbuster"
    else: return "All-Time Blockbuster"

def tmdb_score(vote_avg, vote_count):
    return vote_avg * np.log1p(vote_count)
    
def to_str(x):
    return str(x)




def preprocessed(movie):
    movie.dropna(inplace=True)
    movie['imdb_rating']=round(movie['imdb_rating'],1)
    movie['popularity']=round(movie['popularity'],1)
    movie['imdb_rating'] = movie['imdb_rating'].round(1)
    movie['release_date'] = movie['release_date'].apply(fetch_year_decade)
    movie['cast'] = movie['cast'].apply(cast_fetching)
    
    movie['imdb_score'] = movie.apply(lambda row: imdb_score(row['imdb_rating'], row['imdb_votes']), axis=1)
    movie['imdb_score']= round(movie['imdb_score'],2)
    movie['financial_group'] = movie.apply(lambda row: financial_group(row['revenue'], row['budget']), axis=1)
    movie['tmdb_score'] = movie.apply(lambda row: tmdb_score(row['vote_average'], row['vote_count']), axis=1)
    movie['tmdb_score']= round(movie['tmdb_score'],2)
    movie = movie.drop(columns=['imdb_rating','imdb_votes','vote_average','vote_count','budget','revenue'])
    
    movie=movie.applymap(to_str)
    # List only the columns where you want spaces completely removed
    cols_to_strip = ['cast', 'director', 'genres']
    
    # Apply the space removal ONLY to those specific columns
    for col in cols_to_strip:
        movie[col] = movie[col].astype(str).str.replace(" ", "")
    
    # Handle your countries safely and separately
    movie["production_countries"] = movie["production_countries"].astype(str).str.replace(" ", "").str.replace(",", " ")    
    return movie

preprocessing

In [18]:
movie = preprocessed(movie)
movie.head()


C:\Users\91942\AppData\Local\Temp\ipykernel_20848\2062003404.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  movie=movie.applymap(to_str)


,id,title,release_date,imdb_id,original_language,overview,popularity,genres,production_companies,production_countries,cast,director,poster_path,imdb_score,financial_group,tmdb_score
0,2,Ariel,198,tt0094675,fi,A Finnish man goes to the city to find a job a...,1.6,"Comedy,Drama,Romance,Crime",Villealfa Filmproductions,Finland,Kar,AkiKaurismäki,/ojDg0PGvs6R9xYFodRct2kdI6wC.jpg,68.0,None,42.06
1,3,Shadows in Paradise,198,tt0092149,fi,"Nikander, a rubbish collector and would-be ent...",1.8,"Comedy,Drama,Romance",Villealfa Filmproductions,Finland,Juk,AkiKaurismäki,/nj01hspawPof0mJmlgfjuLyJuRN.jpg,67.11,None,44.43
2,5,Four Rooms,199,tt0113101,en,It's Ted the Bellhop's first night on the job....,3.6,Comedy,"Miramax, A Band Apart",UnitedStatesofAmerica,Sam,"QuentinTarantino,RobertRodriguez,AlexandreRock...",/75aHn1NOYXh4M7L5shoeQ6NGykP.jpg,78.19,Average,46.98
3,6,Judgment Night,199,tt0107286,en,"Four young friends, while taking a shortcut en...",2.2,"Action,Crime,Thriller","Largo Entertainment, JVC, Universal Pictures",UnitedStatesofAmerica,Jer,StephenHopkins,/3rvvpS9YPM5HB2f4HYiNiJVtdam.jpg,65.72,Flop,38.46
6,11,Star Wars,197,tt0076759,en,Princess Leia is captured and held hostage by ...,24.9,"Adventure,Action,ScienceFiction",Lucasfilm Ltd.,UnitedStatesofAmerica,Mic,GeorgeLucas,/6FfCtAuVAW8XJjZ7eWeLibRLWTw.jpg,122.71,All-Time Blockbuster,82.18


In [19]:
movie["production_countries"][11]

'Denmark Finland France Germany Iceland Netherlands Norway Sweden'

all tags combine then it will call corpas

reduce tag 

In [24]:
import nltk

ModuleNotFoundError: No module named 'nltk'

In [ ]:
from nltk.stem.porter import PorterStemmer

ps=PorterStemmer()

In [ ]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. YOUR FAST MATH FUNCTION ---
def get_top_k_recommendations_dict(X, chunk_size=100, top_k=10):
    n_samples = X.shape[0]
    recommendations_dict = {}
    actual_k = min(top_k + 1, n_samples) # +1 to account for self-match
    
    for i in range(0, n_samples, chunk_size):
        end_idx = min(i + chunk_size, n_samples)
        chunk_sim = cosine_similarity(X[i:end_idx], X)
        
        # Your original argpartition logic for maximum speed
        unsorted_idx = np.argpartition(chunk_sim, -actual_k, axis=1)[:, -actual_k:]
        top_vals = np.take_along_axis(chunk_sim, unsorted_idx, axis=1)
        sort_order = np.argsort(-top_vals, axis=1)
        sorted_idx = np.take_along_axis(unsorted_idx, sort_order, axis=1)
        
        # Build the dictionary with (Index, Score) tuples
        for row_idx, actual_movie_idx in enumerate(range(i, end_idx)):
            rec_tuples = [
                (idx, score) for idx, score in zip(sorted_idx[row_idx], top_vals[row_idx])
                if idx != actual_movie_idx # Remove self-match
            ]
            recommendations_dict[actual_movie_idx] = rec_tuples[:top_k]
            
    return recommendations_dict

# --- 2. YOUR PANDAS PROCESSING FUNCTION ---
def get_movie_for_given_lang(movie):
    # Keep only needed columns
    new_df = movie[['id', 'release_date', 'original_language', 'popularity', 'genres',
                    'production_companies', 'production_countries', 'cast', 'director',
                    'imdb_score', 'financial_group', 'tmdb_score', 'overview']].copy()
    
    # Apply your 3x weight to important columns
    imp_cols = ['original_language', 'director', 'cast', 'genres', 'release_date']
    for col in imp_cols:
        new_df[col] = (new_df[col].fillna('').astype(str) + ' ') * 3
        
    # Build final 'tag' column, lowercase, and stem
    new_df['tag'] = new_df.astype(str).agg(' '.join, axis=1).str.lower()
    new_df['tag'] = new_df['tag'].apply(stem)
    new_df = new_df.dropna(subset=['tag'])
    
    # Vectorize
    vectorizer = TfidfVectorizer(stop_words='english', max_features=8000, decode_error='replace')
    X = vectorizer.fit_transform(new_df['tag'])
    
    # Get mathematical recommendations
    top_matches_dict = get_top_k_recommendations_dict(X, chunk_size=100, top_k=10)
    
    # Your exact mapping logic: Convert Row Index -> Movie ID
    movie_idx = {}
    for key, value in top_matches_dict.items():
        main_movie_id = new_df['id'].iloc[key]
        
        # Unpack tuple, lookup ID via .iloc, and round score
        movie_idx[main_movie_id] = [
            (new_df['id'].iloc[row_idx], float(round(score, 4))) for row_idx, score in value
        ]
        
    return movie_idx

In [ ]:
def get_name(idx):
    # Find the title
    name = movie.loc[movie["id"] == idx, "title"]
    
    # Check if a movie was actually found to prevent errors
    if not name.empty:
        # Extract the raw text and RETURN it (do not print it here!)
        return name.iloc[0]
    else:
        return "Unknown Movie"

In [ ]:
import pickle


In [ ]:
langs

['UnitedStatesofAmerica',
 'France',
 'UnitedKingdom',
 'Japan',
 'Germany',
 'Canada',
 'India',
 'Italy',
 'Brazil',
 'Spain',
 'Mexico',
 'Russia',
 'China',
 'SovietUnion',
 'Argentina']

In [ ]:
def movie_recommend_for_lang(movie, langs):
  
    all_country_movies = {}
    
    for country in langs:
        
        country_movie = movie[movie["production_countries"].str.contains(country, na=False)]
        print(f"Calculating metrics for {country}...")
        
        country_metrics = get_movie_for_given_lang(country_movie)

        all_country_movies[country] = country_movie

        temp_filename = f"temp_{country}.pkl"
        with open(temp_filename, 'wb') as f:
            pickle.dump(country_metrics, f, protocol=pickle.HIGHEST_PROTOCOL)
        
        print(f" Safely checkpointed {country} to {temp_filename}!\n")
    return all_country_movies

In [ ]:
langs_8 = langs[:7]
all_dict = movie_recommend_for_lang(movie,langs_8)

Calculating metrics for UnitedStatesofAmerica...
 Safely checkpointed UnitedStatesofAmerica to temp_UnitedStatesofAmerica.pkl!

Calculating metrics for France...
 Safely checkpointed France to temp_France.pkl!

Calculating metrics for UnitedKingdom...
 Safely checkpointed UnitedKingdom to temp_UnitedKingdom.pkl!

Calculating metrics for Japan...
 Safely checkpointed Japan to temp_Japan.pkl!

Calculating metrics for Germany...
 Safely checkpointed Germany to temp_Germany.pkl!

Calculating metrics for Canada...
 Safely checkpointed Canada to temp_Canada.pkl!

Calculating metrics for India...
 Safely checkpointed India to temp_India.pkl!



In [ ]:
import os, pickle

my_langs = langs_8
master_dict = {}

# 1. Merge all temp files and delete them
for country in my_langs:
    filename = f"temp_{country}.pkl"
    try:
        # Try to open the file and add it to the master dictionary
        with open(filename, 'rb') as f:
            master_dict[country] = pickle.load(f)
            
        # Delete the temp file immediately after loading it
        os.remove(filename) 
        
    except FileNotFoundError:
        # If the file doesn't exist (because of a crash earlier), just skip it silently
        pass 

# 2. Save the final combined file
with open('all_countries_metrics.pkl', 'wb') as f:
    pickle.dump(master_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Merge complete! Master file saved.")

Merge complete! Master file saved.


In [ ]:
with open('all_movie.pkl', 'wb') as f:
            pickle.dump(movie, f, protocol=pickle.HIGHEST_PROTOCOL)